# Consolidated Evaluation Harness
Your material covers each metric separately. Expert practice means running them **together on the same dataset** to compare retrievers head-to-head. This notebook computes Precision@K, Recall@K, MRR, and NDCG side by side, then compares two retrievers on identical queries. Fully offline.

In [ ]:
# Self-contained mock embedder + LLM so this notebook runs with NO api key / NO network.
# Swap MockEmbedder -> your InHouseEmbeddings and mock_llm -> your ask() for real use.
import numpy as np, re
from collections import Counter

class MockEmbedder:
    """Deterministic bag-of-words embedding: same words -> similar vectors.
    Good enough to demonstrate retrieval behavior without a real model."""
    def __init__(self, dim=64):
        self.dim = dim
    def _vec(self, text):
        rng = np.random.default_rng(0)
        base = {}
        v = np.zeros(self.dim)
        for w in re.findall(r"\w+", text.lower()):
            h = abs(hash(w)) % self.dim
            v[h] += 1.0
        n = np.linalg.norm(v)
        return v / n if n > 0 else v
    def embed_documents(self, texts): return [self._vec(t).tolist() for t in texts]
    def embed_query(self, text): return self._vec(text).tolist()

embedder = MockEmbedder()

def mock_llm(system, user, **kw):
    """Extremely dumb stand-in: echoes retrieved context if present, else says IDK.
    Replace with your real ask(system, user, model=...) helper for genuine answers."""
    if "no relevant" in user.lower() or "context: \n\nquestion" in user.lower():
        return "I don't know based on the provided context."
    # pull the 'Context:' block back out as a fake 'answer'
    m = re.search(r"Context:(.*?)Question:", user, re.S)
    ctx = m.group(1).strip() if m else ""
    return f"(mock answer grounded in retrieved context) {ctx[:160]}"

## A labeled mini-benchmark (queries + known-relevant doc ids)

In [ ]:
import numpy as np

CORPUS = {
    "d1": "Refunds are processed within 5 business days.",
    "d2": "Refund requests require an order number and reason.",
    "d3": "Standard shipping takes 3 to 5 business days.",
    "d4": "Express shipping is delivered next day.",
    "d5": "Warranty covers manufacturing defects for 12 months.",
    "d6": "Warranty claims need the original receipt.",
}
ids = list(CORPUS); texts = list(CORPUS.values())
mat = np.array(embedder.embed_documents(texts))

# gold: for each query, the set of truly-relevant doc ids
BENCHMARK = [
    {"query": "how long do refunds take",      "relevant": {"d1", "d2"}},
    {"query": "when will my shipment arrive",   "relevant": {"d3", "d4"}},
    {"query": "what does the warranty cover",   "relevant": {"d5", "d6"}},
]

def retrieve_ids(query, k):
    q = np.array(embedder.embed_query(query))
    order = np.argsort(mat @ q)[::-1][:k]
    return [ids[i] for i in order]

## All four retrieval metrics in one place

In [ ]:
def precision_at_k(retrieved, relevant, k):
    top = retrieved[:k]
    return len(set(top) & relevant) / k

def recall_at_k(retrieved, relevant, k):
    top = retrieved[:k]
    return len(set(top) & relevant) / len(relevant)

def reciprocal_rank(retrieved, relevant):
    for rank, doc in enumerate(retrieved, 1):
        if doc in relevant:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(retrieved, relevant, k):
    dcg = sum((1.0 / np.log2(rank + 1)) for rank, doc in enumerate(retrieved[:k], 1) if doc in relevant)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0

K = 3
print(f"{'query':<32} P@{K}  R@{K}   RR    NDCG@{K}")
agg = {"p": [], "r": [], "rr": [], "ndcg": []}
for item in BENCHMARK:
    ret = retrieve_ids(item["query"], K)
    rel = item["relevant"]
    p, r = precision_at_k(ret, rel, K), recall_at_k(ret, rel, K)
    rr, nd = reciprocal_rank(ret, rel), ndcg_at_k(ret, rel, K)
    agg["p"].append(p); agg["r"].append(r); agg["rr"].append(rr); agg["ndcg"].append(nd)
    print(f"{item['query']:<32} {p:.2f}  {r:.2f}  {rr:.2f}  {nd:.2f}")

print(f"\n{'MEAN':<32} {np.mean(agg['p']):.2f}  {np.mean(agg['r']):.2f}  "
      f"{np.mean(agg['rr']):.2f}  {np.mean(agg['ndcg']):.2f}")
print("(mean RR across queries = MRR)")

## Head-to-head: compare two retrievers on identical queries
Here we compare k=1 vs k=3 retrieval. In real use, swap one for a different embedding model or a hybrid retriever — the harness stays identical.

In [ ]:
def evaluate(retriever_fn, k):
    scores = {"p": [], "r": [], "rr": [], "ndcg": []}
    for item in BENCHMARK:
        ret = retriever_fn(item["query"], k)
        rel = item["relevant"]
        scores["p"].append(precision_at_k(ret, rel, k))
        scores["r"].append(recall_at_k(ret, rel, k))
        scores["rr"].append(reciprocal_rank(ret, rel))
        scores["ndcg"].append(ndcg_at_k(ret, rel, k))
    return {m: np.mean(v) for m, v in scores.items()}

r_k1 = evaluate(retrieve_ids, 1)
r_k3 = evaluate(retrieve_ids, 3)
print("Retriever A (k=1):", {m: round(v, 3) for m, v in r_k1.items()})
print("Retriever B (k=3):", {m: round(v, 3) for m, v in r_k3.items()})
print("\nExpert takeaway: k=1 often wins precision but loses recall. The 'best' k")
print("depends on whether your downstream LLM handles noise well (favor recall)")
print("or gets distracted by irrelevant chunks (favor precision).")

## Your turn
1. Swap `MockEmbedder` for your real embedder and re-run — do the metrics change rank order?
2. Add Answer Faithfulness: generate an answer per query with your real `ask()`, then use a second LLM call as judge (score 1-5 whether the answer only uses retrieved context).
3. Expand the benchmark to 15-20 queries — small benchmarks are noisy; expert eval needs volume.